In [3]:
!pip install -q google-cloud-documentai
!pip install -q PyMuPDF
!pip install -q google-cloud-storage
!pip show -q google-cloud-documentai
!pip install -q --upgrade google-cloud-documentai

In [12]:
from typing import Optional

from google.api_core.client_options import ClientOptions
from google.cloud import documentai  # type: ignore

def process_document_sample(
    project_id: str,
    location: str,
    processor_id: str,
    file_path: str,
    mime_type: str,
    field_mask: Optional[str] = None,
    processor_version_id: Optional[str] = None,
    
) -> None:
    # You must set the `api_endpoint` if you use a location other than "us".
    opts = ClientOptions(api_endpoint=f"{location}-documentai.googleapis.com")
  
    client = documentai.DocumentProcessorServiceClient(client_options=opts)

    if processor_version_id:
        # The full resource name of the processor version, e.g.:
        #  projects/552196031736/locations/us/processors/dd0d45f5c877fcca:process
        # `projects/{project_id}/locations/{location}/processors/{processor_id}/processorVersions/{processor_version_id}`
        name = client.processor_version_path(
            project_id, location, processor_id, processor_version_id
        
        )
    else:
        # The full resource name of the processor, e.g.:
        # `projects/{project_id}/locations/{location}/processors/{processor_id}`
        name = client.processor_path(project_id, location, processor_id)
      
    # Read the file into memory
    with open(file_path, "rb") as image:
        image_content = image.read()

    # Load binary data
    raw_document = documentai.RawDocument(content=image_content, mime_type=mime_type)

    # For more information: https://cloud.google.com/document-ai/docs/reference/rest/v1/ProcessOptions
    # Optional: Additional configurations for processing.
    process_options = documentai.ProcessOptions(
        # Process only specific pages
        individual_page_selector=documentai.ProcessOptions.IndividualPageSelector(
            pages=[1]
        )
    )

    # Configure the process request
    request = documentai.ProcessRequest(
        name=name,
        raw_document=raw_document,
        field_mask=field_mask,
        process_options=process_options,
    )

    result = client.process_document(request=request)

    # For a full list of `Document` object attributes, reference this page:
    # https://cloud.google.com/document-ai/docs/reference/rest/v1/Document
    document = result.document

    # Read the text recognition output from the processor
    print("The document contains the following text:")
    print(document.entities)


In [16]:
process_document_sample(
  project_id="",
  location="",
  processor_id="",
  file_path="/home/jupyter/doc_parsing/1.pdf",
  mime_type="application/pdf",
  #field_mask = 'document_name,set_no,responding_party,propounding_party,case_number'

)

The document contains the following text:
[text_anchor {
  text_segments {
    start_index: 605
    end_index: 616
  }
}
type_: "responding_party"
mention_text: "CHRIS BOWEN"
confidence: 1.0
page_anchor {
  page_refs {
    bounding_poly {
      normalized_vertices {
        x: 0.20477814972400665
        y: 0.3138461410999298
      }
      normalized_vertices {
        x: 0.3361774682998657
        y: 0.3138461410999298
      }
      normalized_vertices {
        x: 0.3361774682998657
        y: 0.32659339904785156
      }
      normalized_vertices {
        x: 0.20477814972400665
        y: 0.32659339904785156
      }
    }
  }
}
id: "0"
, text_anchor {
  text_segments {
    start_index: 2765
    end_index: 2777
  }
}
type_: "case_number"
mention_text: "CIVDS9125858"
confidence: 1.0
page_anchor {
  page_refs {
    bounding_poly {
      normalized_vertices {
        x: 0.7377701997756958
        y: 0.3160439431667328
      }
      normalized_vertices {
        x: 0.8623435497283936
   